# 01 — ROI and parcel selection

Select study ROIs (mask ∩ Schaefer-1000), sort by Yeo-7 network, and write `analysis_metadata.json` for downstream notebooks (aligned with `graph_theory` notebook `01_config_and_data_loading.ipynb`).


In [1]:
import os
from pathlib import Path
import yaml

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

    with open(PROJECT_ROOT / "config.yaml", encoding="utf-8") as f:
        CONFIG = yaml.safe_load(f)

        DATA_MODE = CONFIG.get("data_mode", "example")
        N_STATES = CONFIG["n_states"]
        N_TIMEPOINTS = CONFIG["n_timepoints"]
        N_SUBJECTS = CONFIG["n_subjects"]
        METHOD = CONFIG["method"]
        USE_GAMMA = CONFIG["use_gamma"]
        MIN_EFF_N = CONFIG["min_eff_n"]
        MIN_HARD_N = CONFIG["min_hard_n"]
        GRAD = CONFIG["gradient"]

        DATA_ROOT = PROJECT_ROOT / (CONFIG["paths"]["example_root"] if DATA_MODE == "example" else CONFIG["paths"]["study_data_root"])
        OUTPUT_ROOT = PROJECT_ROOT / CONFIG["paths"]["output_root"] / DATA_MODE
        OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

        print(f"project={PROJECT_ROOT} mode={DATA_MODE} out={OUTPUT_ROOT}")

        import numpy as np
        import pandas as pd

        out_dir = OUTPUT_ROOT / "01_roi"
        out_dir.mkdir(parents=True, exist_ok=True)

        if DATA_MODE == "example":
            labels_df = pd.read_csv(DATA_ROOT / "network_labels.csv")
            select_networks = labels_df["network"].values
        else:
            import nibabel as nib
            from nilearn.image import math_img
            from nilearn.datasets import fetch_atlas_schaefer_2018
            atlas = fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=7, resolution_mm=2)
            merged = nib.load(PROJECT_ROOT / CONFIG["paths"]["study_mask"])
            inter = math_img("img1 * img2", img1=merged, img2=nib.load(atlas.maps))
            ulabels = np.unique(inter.get_fdata()[inter.get_fdata() > 0])
            net_names = pd.read_csv(atlas.labels, sep="\t", header=None)[1].str.split("_", expand=True)[2].values
            select_networks = np.array([net_names[int(l)-1] for l in ulabels])
            vc, cn = np.unique(select_networks, return_counts=True)
            select_networks = select_networks[np.isin(select_networks, vc[cn >= 3])]

            sort_idx = np.argsort(select_networks)
            sorted_networks = select_networks[sort_idx]
            pd.DataFrame({"network": sorted_networks}).to_csv(out_dir / "sorted_networks.csv", index=False)
            np.save(out_dir / "sort_idx.npy", sort_idx)

            import json
            subjects = (
            pd.read_csv(DATA_ROOT / "subject_indices.csv")["subject"].tolist()
            if DATA_MODE == "example"
            else []
            )
            meta = {
            "data_mode": DATA_MODE,
            "use_example_data": DATA_MODE == "example",
            "K": N_STATES,
            "method": METHOD,
            "n_timepoints": N_TIMEPOINTS,
            "n_subjects": N_SUBJECTS,
            "use_gamma": USE_GAMMA,
            "subjects": subjects,
            "select_networks": sorted_networks.tolist(),
            }
            meta_path = OUTPUT_ROOT / "analysis_metadata.json"
            with open(meta_path, "w", encoding="utf-8") as f:
                json.dump(meta, f, ensure_ascii=False, indent=2)
                print(f"Saved {len(sorted_networks)} ROIs → {out_dir}")
                print(f"Saved metadata → {meta_path}")

project=d:\文件同步\文件\数据分析结果\data_code_figure\gradient_dispersion_mediate_moderate\gradient_analysis mode=example out=d:\文件同步\文件\数据分析结果\data_code_figure\gradient_dispersion_mediate_moderate\gradient_analysis\results\example


Saved 35 ROIs → d:\文件同步\文件\数据分析结果\data_code_figure\gradient_dispersion_mediate_moderate\gradient_analysis\results\example\01_roi
Saved metadata → d:\文件同步\文件\数据分析结果\data_code_figure\gradient_dispersion_mediate_moderate\gradient_analysis\results\example\analysis_metadata.json
